# Cycle topology - RQ2 campaign

Ring network (0-1-2-3-0): every agent has exactly **2** neighbours, so it is
degree-regular like the clique but sparse like the line. That separates
*symmetry* from *connectivity*, which the star cannot.

**Before running:** Accelerator = `GPU T4 x2`, Internet = `On`, and a Kaggle
Secret named `HF_TOKEN` for gated models (Gemma, Llama).

Pick `MODEL_ID` and `SESSION` in the config cell; everything else is derived.
Each model needs two sessions (A then B) except Qwen3-4B, which fits in one.
For runs over ~2 h use **Save Version -> Save & Run All**, so a browser
disconnect cannot kill the session.


In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch, transformers
assert torch.cuda.is_available(), "GPU not enabled! Settings -> Accelerator -> GPU T4"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU:  ", torch.cuda.get_device_name(0), f"({gb:.1f} GB)")
print("torch:", torch.__version__, " transformers:", transformers.__version__)


In [ ]:
GITHUB_REPO = "https://github.com/stsimpe/cheaptalk_bench.git"

import os, subprocess
if os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", GITHUB_REPO, "/kaggle/working/repo"], check=True)
%cd /kaggle/working/repo
head = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print("HEAD:", head.strip())


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HUGGINGFACE_API_KEY"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded (needed for Gemma / Llama)")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)


## Config - the only cell you edit

`MAX_NEW_TOKENS` is matched to each model's **star** setting so the
star-vs-cycle contrast carries no extra confound. Do not "tidy" these numbers.


In [ ]:
MODEL_ID = "google/gemma-2-2b-it"
# MODEL_ID = "Qwen/Qwen3-4B"               # only model whose A+B fit in one session
# MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
# MODEL_ID = "google/gemma-2-9b-it"

SESSION  = "A"      # A = baseline+no_sense+silence+counterfactual, B = the 3 framings
TOPOLOGY = "cycle"
N_RUNS, N_ROUNDS = 5, 16      # protocol rule: 5 runs everywhere

SCENARIOS = {
    "A": ["baseline", "no_sense", "silence", "counterfactual"],
    "B": ["framing_business", "framing_team", "framing_competitive"],
}[SESSION]

# max_new_tokens per model, per session, copied from the star campaign.
MAX_NEW_TOKENS = {
    "meta-llama/Llama-3.1-8B-Instruct": {"A": 192, "B": 192},
    "Qwen/Qwen2.5-7B-Instruct":         {"A": 512, "B": 256},
    "Qwen/Qwen3-4B":                    {"A": 256, "B": 256},
    "google/gemma-2-2b-it":             {"A": 256, "B": 192},
    "google/gemma-2-9b-it":             {"A": 160, "B": 192},
}[MODEL_ID][SESSION]

# runs the protocol demands: every scenario is 2 games x N_RUNS, and baseline
# additionally has a no_comm arm.
EXPECTED_RUNS = len(SCENARIOS) * 2 * N_RUNS + (2 * N_RUNS if "baseline" in SCENARIOS else 0)

MODEL_SHORT = MODEL_ID.split("/")[-1]
OUT_DIR_BASE = "/kaggle/working/results/" + MODEL_SHORT
# run_all_scenarios.py appends the topology suffix itself for non-star runs:
RESULT_DIR = OUT_DIR_BASE if TOPOLOGY == "star" else OUT_DIR_BASE + "_" + TOPOLOGY

print("Model     :", MODEL_ID)
print("Session " + SESSION + " :", SCENARIOS)
print("Topology  :", TOPOLOGY, "runs=", N_RUNS, "rounds=", N_ROUNDS, "max_new_tokens=", MAX_NEW_TOKENS)
print("Expecting :", EXPECTED_RUNS, "run files")
print("Results ->:", RESULT_DIR)


## Run

The command is built as an argument list and executed with `subprocess`,
deliberately **not** as `!python ... $VAR`. Shell magic expands an undefined or
misspelled variable to an empty string, which kills the run with
`error: argument --model-id: expected one argument` while every print above it
still looks correct, and then lets the notebook package an empty zip. A
non-zero exit code raises here instead.


In [ ]:
import subprocess, sys, shlex, time

cmd = [
    sys.executable, "run_all_scenarios.py",
    "--provider", "local",
    "--model-id", MODEL_ID,
    "--topology", TOPOLOGY,
    "--n-runs", str(N_RUNS),
    "--n-rounds", str(N_ROUNDS),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--out-dir-base", OUT_DIR_BASE,
    "--zip-mirror", "/kaggle/working",
    "--no-probe",
    "--scenarios", *SCENARIOS,
]
print("RUN:", " ".join(shlex.quote(c) for c in cmd), flush=True)

t0 = time.time()
rc = subprocess.run(cmd).returncode
elapsed = time.time() - t0
print()
print("exit code", rc, "after", round(elapsed / 60, 1), "min",
      "(" + str(round(elapsed / 3600, 2)) + " h)")
if rc != 0:
    raise SystemExit("run_all_scenarios.py FAILED (exit " + str(rc) + "). Nothing to package.")


## Verify, then package

Counts the runs actually produced and checks the topology recorded inside the
files, so a half-finished session is visible now rather than three weeks later
in the analysis.


In [ ]:
import glob, json, os, shutil
from collections import Counter

files = [f for f in glob.glob(os.path.join(RESULT_DIR, "**", "*.json"), recursive=True)
         if not os.path.basename(f).startswith("_progress")]

per_scenario = Counter()
topologies = Counter()
for f in files:
    rec = json.load(open(f))
    per_scenario[os.path.relpath(f, RESULT_DIR).split(os.sep)[0]] += 1
    topologies[rec["topology"]["type"]] += 1

print("runs found:", len(files), " expected:", EXPECTED_RUNS)
print("topologies:", dict(topologies))
for k in sorted(per_scenario):
    print("   ", k.ljust(24), per_scenario[k], "runs")

assert files, "No runs were produced."
assert set(topologies) == {TOPOLOGY}, "Wrong topology recorded: " + str(dict(topologies))
if len(files) != EXPECTED_RUNS:
    print()
    print("WARNING: expected", EXPECTED_RUNS, "runs but found", len(files),
          "- a scenario is incomplete.")

zip_base = "/kaggle/working/" + MODEL_SHORT + "_" + TOPOLOGY + "_session" + SESSION
shutil.make_archive(zip_base, "zip", RESULT_DIR)
size_mb = os.path.getsize(zip_base + ".zip") / 1e6
print()
print("Download this:", zip_base + ".zip", "(" + str(round(size_mb, 1)) + " MB)")


## After downloading

1. Put the zip in `diplomatikh/drive_sync/` and upload it to the Drive folder
   `cheaptalk_bench_results`.
2. Append the runs to the `Runs` sheet of `cheaptalk_results_tracker.xlsx`;
   `Combinations` and `Results` recompute themselves.
3. Add a row to `TRACK_RECORD.md`, noting the wall-clock time printed by the run
   cell so the remaining estimates can be recalibrated.
